In [ ]:
# ============================================================
# CELL 1: SETUP & CONFIG
# ============================================================
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["PYDEVD_DISABLE_FILE_VALIDATION"] = "1"
os.environ["PYTHONWARNINGS"] = "ignore"
import sys
sys.stderr = open(os.devnull, "w")

print("Installing packages...")
!pip install -q transformers accelerate faiss-cpu rank-bm25 sentence-transformers > /dev/null
!pip install -q bitsandbytes pyarrow scikit-learn matplotlib pandas numpy tqdm > /dev/null
from IPython.display import clear_output
clear_output(wait=True)
print("\nPackages installed successfully")

import os, json, time, pickle, re, gc, warnings, random
from pathlib import Path
from datetime import datetime

import torch
import faiss
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel, BitsAndBytesConfig
from sentence_transformers import CrossEncoder
from rank_bm25 import BM25Okapi
from sklearn.model_selection import train_test_split
from IPython.display import display, HTML
warnings.filterwarnings("ignore")

def clean_text(text):
    if not isinstance(text, str): return ""
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
random.seed(SEED)

# -------------------------- ABLATION STUDY --------------------------
# Chỉnh sửa biến này để chạy từng kịch bản Ablation:
# 1: BM25 Only
# 2: Qwen3 Only (Dense + FAISS) - Gộp case 2 & 3
# 3: Qwen3 + BM25 (Hybrid)
# 4: Qwen3 + BM25 + MMR
# 5: Full Pipeline (Qwen3 + BM25 + MMR + Reranker)
ABLATION_MODE = 1

USE_FAISS    = ABLATION_MODE >= 2
USE_BM25     = ABLATION_MODE == 1 or ABLATION_MODE >= 3
USE_MMR      = ABLATION_MODE >= 4
USE_RERANKER = ABLATION_MODE == 5

# -------------------------- CẤU HÌNH TỐI ƯU --------------------------
DATA_SIZE       = 3_000_000          
MAX_READ_LINES  = 6_000_000
MIN_RATING      = 3.5
PRODUCT_LIMIT   = 10_000            

MAX_LEN               = 256        
BATCH_SIZE_TOTAL      = 120
MODEL_NAME            = "Qwen/Qwen3-Embedding-8B"
RERANKER_MODEL        = "cross-encoder/ms-marco-MiniLM-L-12-v2" 
USE_8BIT              = True

TOP_K_RETRIEVE   = 200
TOP_K_FINAL      = 5
HYBRID_ALPHA     = 0.6
MMR_LAMBDA       = 0.5             # Trọng số cho MMR
EVAL_CANDIDATE_K = 200             
EVAL_SAMPLE_SIZE = 300             

FORCE_RELOAD_EMBED = False
USE_CHECKPOINT     = True

TRAIN_RATIO = 0.8
TEST_RATIO  = 0.2

REVIEW_PATH = "/kaggle/input/datasets/loilehuu/automotive-dataset/Automotive.jsonl"
META_PATH   = "/kaggle/input/datasets/loilehuu/automotive-dataset/meta_Automotive.jsonl"
VERSION    = datetime.now().strftime("%Y%m%d_%H%M")
SAVE_DIR   = f"/kaggle/working/recommend{VERSION}"
OUTPUT_DIR = SAVE_DIR
os.makedirs(SAVE_DIR, exist_ok=True)

TRAINED = False

GPU_COUNT = torch.cuda.device_count()
print("SYSTEM INFO".center(110, "="))
print(f"Ready with {GPU_COUNT} GPU.")
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader
print("PATHS + CONFIG".center(110, "="))
print(f"SAVE_DIR: {SAVE_DIR}")
print(f"ABLATION MODE: {ABLATION_MODE} | PRODUCT_LIMIT:{PRODUCT_LIMIT} | FAISS:{USE_FAISS} | BM25:{USE_BM25} | MMR:{USE_MMR} | RERANK:{USE_RERANKER}")

Installing packages...


In [ ]:
# ============================================================
# CELL 2: LOAD & TIỀN XỬ LÝ DỮ LIỆU
# ============================================================
print("LOAD & DATA PREPROCESSING".center(110, "="))
def load_reviews_stratified(path, size, min_rating=3.5, max_read_lines=None, quiet=True):
    data = []
    with open(path, "r") as f:
        for line_num, line in enumerate(tqdm(f, desc="Loading reviews", unit=" lines", disable=quiet)):
            if max_read_lines is not None and line_num >= max_read_lines:
                if not quiet: print(f" Đã đạt giới hạn {max_read_lines} dòng, dừng sớm.")
                break
            if len(data) >= size:
                break
            try:
                item = json.loads(line)
            except json.JSONDecodeError:
                continue
            if float(item.get("rating", 0)) >= min_rating:
                data.append(item)
    df = pd.DataFrame(data)
    seen_asins = {item["parent_asin"] for item in data}
    return df, seen_asins

def load_meta(path, valid_asins):
    meta_data = []
    with open(path, "r") as f:
        for line in tqdm(f, desc="Loading meta"):
            try:
                item = json.loads(line)
            except json.JSONDecodeError:
                continue
            if item["parent_asin"] in valid_asins:
                price_val = item.get("price")
                item["price"] = str(price_val) if price_val is not None else ""
                meta_data.append(item)
    df = pd.DataFrame(meta_data)
    df["brand"] = df["details"].apply(lambda x: x.get("Brand", "Unknown") if isinstance(x, dict) else "Unknown")
    df["description"] = df["description"].apply(lambda x: " ".join(x)[:1500] if isinstance(x, list) else str(x)[:1500])
    df["category"] = df["categories"].apply(lambda x: x[1] if isinstance(x, list) and len(x) > 1 else (x[0] if x else "Other"))
    df["sub_category"] = df["categories"].apply(lambda x: x[-1] if isinstance(x, list) and len(x) > 0 else "Other")
    df["price"] = df["price"].astype(str).replace("nan", "")
    if "features" in df.columns:
        df["features_text"] = df["features"].apply(lambda x: " ".join(x[:5]) if isinstance(x, list) else "")
    else:
        df["features_text"] = ""
    return df[["parent_asin", "title", "description", "brand",
               "category", "sub_category", "price", "features_text"]]

def build_product_df(review_df, meta_df):
    def agg_reviews(texts):
        selected = [t[:200] for t in texts[:3] if len(t.strip()) > 10]
        return " | ".join(selected)
    product_df = review_df.groupby("parent_asin").agg(
        text=("text", agg_reviews),
        rating=("rating", "mean"),
        rating_count=("rating", "count"),
        title=("title", "first")
    ).reset_index()
    product_df = product_df.merge(meta_df, on="parent_asin", how="inner", suffixes=("_review", "_meta"))
    product_df["title"] = product_df["title_meta"].fillna(product_df["title_review"])
    product_df = product_df.drop(columns=["title_review", "title_meta"], errors="ignore").fillna("")
    cat_counts = product_df["category"].value_counts()
    valid_cats = cat_counts[cat_counts >= 3].index
    product_df = product_df[product_df["category"].isin(valid_cats)]
    print(f"Products sau lọc: {len(product_df)}")
    return product_df.reset_index(drop=True)

if not TRAINED:
    review_df, review_asins = load_reviews_stratified(
        REVIEW_PATH, DATA_SIZE, MIN_RATING, max_read_lines=MAX_READ_LINES, quiet=True)
    print(f"Reviews: {len(review_df)}, unique ASINs: {len(review_asins)}")
    meta_df = load_meta(META_PATH, review_asins)
    product_df = build_product_df(review_df, meta_df)

    if len(product_df) > PRODUCT_LIMIT:
        print(f"Sản phẩm vượt giới hạn ({len(product_df)} > {PRODUCT_LIMIT}), lấy mẫu ngẫu nhiên")
        product_df = product_df.sample(n=PRODUCT_LIMIT, random_state=SEED).reset_index(drop=True)
    print(f"Số sản phẩm cuối cùng: {len(product_df)}")
    TRAINED = True

In [ ]:
# ============================================================
# CELL 3: TẠO COMBINED TEXT, SHORT TEXT & SPLIT TRAIN/TEST
# ============================================================
print("CREATE COMBINED TEXT, SHORT TEXT & SPLIT TRAIN/TEST".center(110, "="))
def build_document_text(row):
    features = row.get("features_text", "")[:300]
    features_str = f"\nFeatures: {features}" if features else ""
    return (
        f"Instruct: Represent this automotive product for retrieval.\n"
        f"Category: {row['category']} > {row['sub_category']}\n"
        f"Title: {row['title']}\n"
        f"Brand: {row['brand']}{features_str}\n"
        f"Description: {row['description'][:700]}\n"
        f"Reviews: {row['text'][:400]}"
    )

def build_short_text(row):
    # Nhân 3 title để tăng trọng số + kèm features cho Reranker nhẹ
    title_clean = clean_text(row["title"])
    features_clean = clean_text(row.get("features_text", ""))[:200]
    return f"{title_clean} {title_clean} {title_clean} {features_clean}"

product_df["combined_text"] = product_df.apply(build_document_text, axis=1)
product_df["short_text"] = product_df.apply(build_short_text, axis=1)

product_df = product_df[product_df["combined_text"].str.len() > 20].reset_index(drop=True)
print(f"Sản phẩm sau khi lọc text: {len(product_df)}")

valid_subcats = product_df['sub_category'].value_counts()
valid_subcats = valid_subcats[valid_subcats >= 2].index
product_df = product_df[product_df['sub_category'].isin(valid_subcats)].reset_index(drop=True)

train_df, test_df = train_test_split(
    product_df, test_size=TEST_RATIO, random_state=SEED,
    stratify=product_df['sub_category']
)

# FIX LỖI LỆCH INDEX: Bắt buộc phải reset_index để đồng bộ với output [0,1,2..] của FAISS
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"Train size: {len(train_df)}, Test size: {len(test_df)}")

bm25_corpus = train_df["combined_text"].tolist()
tokenized_corpus = [clean_text(doc).split() for doc in bm25_corpus]
bm25_index = BM25Okapi(tokenized_corpus)

with open(f"{SAVE_DIR}/bm25_index.pkl", "wb") as f:
    pickle.dump(bm25_index, f)
train_df.to_parquet(f"{SAVE_DIR}/train_df.parquet", index=False)
test_df.to_parquet(f"{SAVE_DIR}/test_df.parquet", index=False)
print(f"💾 Đã lưu BM25 index, train_df, test_df vào {SAVE_DIR}")

In [ ]:
# ============================================================
# CELL 4: EMBEDDING 2 GPU SONG SONG + CHECKPOINT 
# ============================================================
print("EMBEDDING 2 GPU".center(110, "="))
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = hf_token
except:
    print(" Không lấy được HF_TOKEN, dùng token mặc định nếu có")

start_time = time.time()
print("🔄 Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print("✅ Tokenizer ready")

def mean_pooling(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).float()
    sum_emb = torch.sum(last_hidden_state * mask, dim=1)
    return sum_emb / torch.clamp(mask.sum(dim=1), min=1e-9)

def load_model_8bit(gpu_id):
    bnb_config = BitsAndBytesConfig(load_in_8bit=True, bnb_8bit_compute_dtype=torch.float16)
    model = AutoModel.from_pretrained(
        MODEL_NAME, device_map={"": f"cuda:{gpu_id}"},
        quantization_config=bnb_config, dtype=torch.float16
    ).eval()
    return model

print("🔄 Loading model into GPU 0 (8-bit)...")
model0 = load_model_8bit(0)
print("✅ GPU 0 ready")
print("🔄 Loading model into GPU 1 (8-bit)...")
model1 = load_model_8bit(1)
print("✅ GPU 1 ready")

def embed_batch(texts, model, device):
    inputs = tokenizer(texts, return_tensors="pt", truncation=True,
                       padding=True, max_length=MAX_LEN)
    inputs = {k: v.to(device, non_blocking=True) for k, v in inputs.items()}
    with torch.inference_mode():
        outputs = model(**inputs)
    emb = mean_pooling(outputs.last_hidden_state, inputs["attention_mask"])
    return emb.float().cpu().numpy()

from concurrent.futures import ThreadPoolExecutor
def get_embedding_parallel(batch_texts):
    n = len(batch_texts)
    if n <= 32:
        return embed_batch(batch_texts, model0, "cuda:0")
    half = n // 2
    with ThreadPoolExecutor(max_workers=2) as ex:
        fut0 = ex.submit(embed_batch, batch_texts[:half], model0, "cuda:0")
        fut1 = ex.submit(embed_batch, batch_texts[half:2*half], model1, "cuda:1")
        emb0, emb1 = fut0.result(), fut1.result()
    res = np.vstack([emb0, emb1])
    if 2*half < n:
        res = np.vstack([res, embed_batch(batch_texts[2*half:], model0, "cuda:0")])
    return res

texts = train_df["combined_text"].tolist()
total = len(texts)
FINAL_EMB_PATH = f"{SAVE_DIR}/embeddings_train.npy"
CHECKPOINT_DIR = Path(SAVE_DIR) / "embed_checkpoints_train"
CHECKPOINT_DIR.mkdir(exist_ok=True)

if os.path.exists(FINAL_EMB_PATH) and not FORCE_RELOAD_EMBED:
    print(f"✅ Đã có embeddings_train.npy, bỏ qua embedding.")
    embeddings_train = np.load(FINAL_EMB_PATH)
else:
    completed_batches = set()
    if USE_CHECKPOINT:
        for ckpt_file in CHECKPOINT_DIR.glob("batch_*.npz"):
            try:
                batch_num = int(ckpt_file.stem.split("_")[1])
                completed_batches.add(batch_num)
            except: pass
        print(f"🔁 Checkpoint: tìm thấy {len(completed_batches)} batch đã hoàn thành.")
    else:
        import shutil
        if CHECKPOINT_DIR.exists():
            shutil.rmtree(CHECKPOINT_DIR)
            CHECKPOINT_DIR.mkdir(exist_ok=True)

    embeddings_list = []
    num_big_batches = (total + BATCH_SIZE_TOTAL - 1) // BATCH_SIZE_TOTAL
    print(f"EMBEDDING {total} PRODUCTS | MAX_LEN={MAX_LEN} | BATCH={BATCH_SIZE_TOTAL}")
    log_interval = 30
    last_log_time = time.time()

    for batch_idx in tqdm(range(num_big_batches), desc="Embedding big batches"):
        batch_num = batch_idx
        if batch_num in completed_batches:
            continue
        start_idx = batch_idx * BATCH_SIZE_TOTAL
        end_idx = min(start_idx + BATCH_SIZE_TOTAL, total)
        batch_texts = texts[start_idx:end_idx]
        emb = get_embedding_parallel(batch_texts)
        if USE_CHECKPOINT:
            np.savez_compressed(
                CHECKPOINT_DIR / f"batch_{batch_num:06d}.npz",
                indices=np.arange(start_idx, end_idx).astype("int32"),
                embeddings=emb.astype("float32"))
        embeddings_list.append(emb)
        torch.cuda.empty_cache()
        current_time = time.time()
        processed = end_idx
        if current_time - last_log_time >= log_interval or processed >= total:
            elapsed_total = current_time - start_time
            speed = processed / (elapsed_total / 60) if elapsed_total > 0 else 0
            vram0 = torch.cuda.memory_allocated(0) / 1e9
            vram1 = torch.cuda.memory_allocated(1) / 1e9
            print(f"⏱️ {processed}/{total} mẫu | {elapsed_total/60:.1f} phút | "
                  f"Speed: {speed:.0f} mẫu/phút | VRAM0: {vram0:.1f}GB | VRAM1: {vram1:.1f}GB")
            last_log_time = current_time

    if USE_CHECKPOINT and len(completed_batches) > 0:
        all_indices, all_emb = [], []
        for ckpt_file in sorted(CHECKPOINT_DIR.glob("batch_*.npz")):
            data = np.load(ckpt_file)
            all_indices.append(data["indices"])
            all_emb.append(data["embeddings"])
        all_indices = np.concatenate(all_indices).astype("int64")
        all_emb = np.vstack(all_emb).astype("float32")
        order = np.argsort(all_indices)
        embeddings_train = all_emb[order]
    else:
        embeddings_train = np.vstack(embeddings_list).astype("float32")

    norms = np.linalg.norm(embeddings_train, axis=1, keepdims=True)
    embeddings_train = embeddings_train / (norms + 1e-8)
    np.save(FINAL_EMB_PATH, embeddings_train)
    print(f"💾 Đã lưu embeddings_train.npy | shape {embeddings_train.shape}")
    if USE_CHECKPOINT:
        import shutil
        shutil.rmtree(CHECKPOINT_DIR)
        print(" Đã xóa thư mục checkpoint.")

print(f"⏱️ Tổng thời gian embedding: {(time.time() - start_time)/60:.1f} phút.")
del model1
gc.collect()
torch.cuda.empty_cache()
query_model = model0
query_tokenizer = tokenizer
print("✅ Đã giữ model0 làm query embedding model.")

In [ ]:
# ============================================================
# CELL 5: XÂY DỰNG FAISS INDEX
# ============================================================
print("BUILD FAISS INDEX".center(110, "="))
import faiss
dimension = embeddings_train.shape[1]
index = faiss.IndexHNSWFlat(dimension, 48)
index.hnsw.efConstruction = 400
index.hnsw.efSearch = 256
index.add(embeddings_train)
print(f"FAISS index: {index.ntotal} vectors (efSearch=256)")
faiss.write_index(index, f"{SAVE_DIR}/faiss_train.index")
print(f"💾 Đã lưu faiss_train.index")

In [ ]:
# ============================================================
# CELL 6: CROSS-ENCODER & HYBRID SEARCH & MMR
# ============================================================
print("CROSS-ENCODER & HYBRID SEARCH & MMR".center(110, "="))
_query_tokenizer = None
_query_model = None
_reranker = None

def get_query_embedder():
    global _query_tokenizer, _query_model
    if 'query_model' in globals() and query_model is not None:
        _query_model = query_model
        _query_tokenizer = query_tokenizer
        return _query_tokenizer, _query_model
    if _query_tokenizer is not None and _query_model is not None:
        return _query_tokenizer, _query_model
    print("🔄 Loading query embedding model (lazy)...")
    _query_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    if _query_tokenizer.pad_token is None:
        _query_tokenizer.pad_token = _query_tokenizer.eos_token
    if USE_8BIT:
        bnb_config = BitsAndBytesConfig(load_in_8bit=True, bnb_8bit_compute_dtype=torch.float16)
        _query_model = AutoModel.from_pretrained(MODEL_NAME, quantization_config=bnb_config, dtype=torch.float16, device_map="auto").eval()
    else:
        _query_model = AutoModel.from_pretrained(MODEL_NAME, dtype=torch.float16, device_map="auto").eval()
    return _query_tokenizer, _query_model

def get_reranker():
    global _reranker
    if _reranker is not None:
        return _reranker
    if not USE_RERANKER:
        return None
    print(f"🔄 Loading CrossEncoder reranker: {RERANKER_MODEL}...")
    device = 'cuda:1' if torch.cuda.device_count() > 1 else 'cuda:0'
    _reranker = CrossEncoder(RERANKER_MODEL, device=device)
    return _reranker

def mean_pooling_query(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state
    mask_expanded = attention_mask.unsqueeze(-1).float()
    sum_embeddings = torch.sum(token_embeddings * mask_expanded, dim=1)
    sum_mask = torch.clamp(mask_expanded.sum(dim=1), min=1e-9)
    return sum_embeddings / sum_mask

def encode_query(query_text):
    tokenizer_q, model_q = get_query_embedder()
    formatted = f"Instruct: Find automotive products matching this description.\nQuery: {query_text}"
    inputs = tokenizer_q(formatted, return_tensors="pt", truncation=True, padding=True, max_length=MAX_LEN)
    device = next(model_q.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.inference_mode():
        outputs = model_q(**inputs)
    emb = mean_pooling_query(outputs, inputs["attention_mask"])
    emb = emb.float().cpu().numpy().astype("float32")
    emb = emb / (np.linalg.norm(emb, axis=1, keepdims=True) + 1e-8)
    return emb

def weighted_rrf_fusion(dense_rank_list=None, sparse_rank_list=None, alpha=HYBRID_ALPHA, k=60):
    scores = {}
    if dense_rank_list is not None:
        for rank, doc_id in enumerate(dense_rank_list):
            scores[doc_id] = scores.get(doc_id, 0) + alpha / (k + rank)
    if sparse_rank_list is not None:
        for rank, doc_id in enumerate(sparse_rank_list):
            scores[doc_id] = scores.get(doc_id, 0) + (1 - alpha) / (k + rank)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)

def apply_mmr(query_emb, candidate_ids, embeddings, lambda_mult=0.5, top_k=5):
    """Tính toán MMR để đa dạng hóa top kết quả"""
    if len(candidate_ids) <= 1: return candidate_ids[:top_k]
    
    cand_embs = embeddings[candidate_ids]
    query_sims = np.dot(cand_embs, query_emb.T).squeeze()
    
    if len(candidate_ids) == 1: return candidate_ids[:top_k]
    doc_sims = np.dot(cand_embs, cand_embs.T)
    
    selected = [np.argmax(query_sims)]
    unselected = list(range(len(candidate_ids)))
    unselected.remove(selected[0])
    
    while len(selected) < top_k and unselected:
        max_mmr = -np.inf
        best_idx = -1
        for idx in unselected:
            mmr_score = lambda_mult * query_sims[idx] - (1 - lambda_mult) * np.max(doc_sims[idx, selected])
            if mmr_score > max_mmr:
                max_mmr = mmr_score
                best_idx = idx
        selected.append(best_idx)
        unselected.remove(best_idx)
        
    return [candidate_ids[i] for i in selected]

def hybrid_retrieve(query, top_k_candidates=EVAL_CANDIDATE_K, ef_search=256):
    dense_ids, sparse_ids = None, None
    if USE_FAISS:
        query_emb = encode_query(query)
        index.hnsw.efSearch = ef_search
        D_sem, I_sem = index.search(query_emb, top_k_candidates * 2)
        dense_ids = I_sem[0].tolist()
    if USE_BM25:
        tokenized_query = clean_text(query).split()
        bm25_scores = bm25_index.get_scores(tokenized_query)
        k_bm25 = min(top_k_candidates * 2, len(bm25_scores))
        I_lex = np.argpartition(bm25_scores, -k_bm25)[-k_bm25:]
        I_lex = I_lex[np.argsort(bm25_scores[I_lex])[::-1]]
        sparse_ids = I_lex.tolist()
        
    if dense_ids is not None and sparse_ids is not None:
        rrf_results = weighted_rrf_fusion(dense_ids, sparse_ids, alpha=HYBRID_ALPHA)
        ranked = [doc_id for doc_id, _ in rrf_results[:top_k_candidates]]
    elif dense_ids is not None: ranked = dense_ids[:top_k_candidates]
    elif sparse_ids is not None: ranked = sparse_ids[:top_k_candidates]
    else: ranked = []
    
    return ranked

def search_pipeline(query, top_k=5, candidate_pool=EVAL_CANDIDATE_K):
    candidates = hybrid_retrieve(query, top_k_candidates=candidate_pool, ef_search=256)
    if not candidates: return pd.DataFrame()
    
    reranker = get_reranker()
    if reranker is not None:
        pairs = [(query, train_df.iloc[idx]["short_text"]) for idx in candidates]
        ce_scores = reranker.predict(pairs, batch_size=64, show_progress_bar=False)
    else:
        ce_scores = np.arange(len(candidates), 0, -1)
        
    ranked = sorted(zip(candidates, ce_scores), key=lambda x: x[1], reverse=True)
    sorted_cands = [idx for idx, _ in ranked]
    
    if USE_MMR:
        query_emb = encode_query(query)
        # Chỉ áp dụng MMR trên tập hẹp (ví dụ top 30) để đảm bảo tốc độ và đa dạng hóa
        final_ids = apply_mmr(query_emb, sorted_cands[:30], embeddings_train, lambda_mult=MMR_LAMBDA, top_k=top_k)
    else:
        final_ids = sorted_cands[:top_k]
        
    return train_df.iloc[final_ids]

print("✅ Search pipeline sẵn sàng (chỉ tìm trong train set)!")
print(f"   FAISS: {USE_FAISS}, BM25: {USE_BM25}, Reranker: {RERANKER_MODEL}, MMR: {USE_MMR}")

In [ ]:
# ============================================================
# CELL 7: TEST NHANH 
# ============================================================
print("QUICK TEST".center(110, "="))
query = "high performance brake pads for sports car"
start = time.time()
results = search_pipeline(query, top_k=5, candidate_pool=EVAL_CANDIDATE_K)
elapsed = time.time() - start
print(f'Query: "{query}"\n')
columns = ["title", "features_text", "category", "rating"]
display(results[columns].rename(columns={"rating": "avg_rating"}))
print(f"\nSearch time: {elapsed:.2f} giây")

In [ ]:
# ============================================================
# CELL 8: EVALUATION VỚI ĐỊNH NGHĨA GROUND TRUTH CHUẨN
# ============================================================
print("EVALUATION WITH GROUND TRUTH".center(110, "="))
import os, json, time, gc
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.metrics import ndcg_score

EVAL_K = 10
EVAL_GT_COL = "sub_category"
EVAL_QUERY_BATCH_SIZE = 16
EVAL_OUTPUT_JSON = os.path.join(OUTPUT_DIR, f"eval_results_mode_{ABLATION_MODE}.json")
EVAL_OUTPUT_FIG  = os.path.join(OUTPUT_DIR, f"metrics_mode_{ABLATION_MODE}.png")

# Từ điển map số mode với tên kịch bản tương ứng
MODE_NAMES = {
    1: "BM25-Only",
    2: "Qwen3-Only(Dense+FAISS)",
    3: "Qwen3+BM25(Hybrid)",
    4: "Qwen3+BM25+MMR",
    5: "Full-Pipeline(Qwen3+BM25+MMR+Reranker)"
}

def hit_rate_at_k(recommended_ids, ground_truth_ids):
    return int(len(set(recommended_ids) & set(ground_truth_ids)) > 0)

def precision_at_k(recommended_ids, ground_truth_ids, k):
    recommended_ids = recommended_ids[:k]
    return len(set(recommended_ids) & set(ground_truth_ids)) / k

def recall_at_k(recommended_ids, ground_truth_ids, k):
    recommended_ids = recommended_ids[:k]
    if len(ground_truth_ids) == 0: return 0.0
    return len(set(recommended_ids) & set(ground_truth_ids)) / len(set(ground_truth_ids))

def ndcg_at_k(recommended_ids, ground_truth_ids, k):
    recommended_ids = recommended_ids[:k]
    y_true = [1 if idx in ground_truth_ids else 0 for idx in recommended_ids]
    if sum(y_true) == 0: return 0.0
    y_score = list(range(len(y_true), 0, -1))
    return ndcg_score([y_true], [y_score], k=k)

def build_query_text(row):
    title = clean_text(str(row.get('title', '')))
    words = title.split()
    query = " ".join(words[:8])
    return query if query else "automotive part"

def get_ground_truth(query_row, train_df):
    query_label = query_row[EVAL_GT_COL]
    query_title_words = set([w for w in clean_text(str(query_row["title"])).split() if len(w) > 2])
    cat_match = train_df[train_df[EVAL_GT_COL] == query_label]
    
    gt_ids = []
    for idx, row in cat_match.iterrows():
        target_title_words = set(clean_text(str(row["title"])).split())
        if len(query_title_words & target_title_words) >= 2:
            gt_ids.append(idx)
            
    if not gt_ids: gt_ids = cat_match.index.tolist()[:15]
    return gt_ids

def batch_encode_queries(query_texts, batch_size=EVAL_QUERY_BATCH_SIZE):
    tokenizer_q, model_q = get_query_embedder()
    device = next(model_q.parameters()).device
    all_embs = []
    for i in tqdm(range(0, len(query_texts), batch_size), desc="Encoding queries", leave=False):
        batch = query_texts[i:i+batch_size]
        formatted = [f"Instruct: Find automotive products matching this description.\nQuery: {q}" for q in batch]
        inputs = tokenizer_q(formatted, return_tensors="pt", truncation=True, padding=True, max_length=MAX_LEN)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.inference_mode():
            outputs = model_q(**inputs)
        emb = mean_pooling_query(outputs, inputs["attention_mask"])
        emb = emb.float().cpu().numpy().astype("float32")
        emb = emb / (np.linalg.norm(emb, axis=1, keepdims=True) + 1e-8)
        all_embs.append(emb)
        del inputs, outputs, emb
        torch.cuda.empty_cache()
        gc.collect()
    return np.vstack(all_embs)

def batch_faiss_search(query_embs, top_k, ef_search=256):
    index.hnsw.efSearch = ef_search
    D, I = index.search(query_embs, top_k)
    return I

def batch_bm25_search(query_texts, top_k):
    sparse_results = []
    for q in tqdm(query_texts, desc="BM25 search", leave=False):
        tokens = clean_text(q).split()
        scores = bm25_index.get_scores(tokens)
        if len(scores) <= top_k:
            top_indices = np.argsort(scores)[::-1]
        else:
            top_indices = np.argpartition(scores, -top_k)[-top_k:]
            top_indices = top_indices[np.argsort(scores[top_indices])[::-1]]
        sparse_results.append(top_indices.tolist())
    return sparse_results

def batch_hybrid_retrieve(query_texts, query_embs, top_k_candidates=EVAL_CANDIDATE_K, ef_search=256):
    n = len(query_texts)
    dense_ids, sparse_ids = None, None
    if USE_FAISS:
        print("  -> FAISS dense retrieval...")
        dense_ids = batch_faiss_search(query_embs, top_k_candidates * 2, ef_search)
    if USE_BM25:
        print("  -> BM25 sparse retrieval...")
        sparse_ids = batch_bm25_search(query_texts, top_k_candidates * 2)
    
    results = []
    for d_ids, s_ids in zip(dense_ids if dense_ids is not None else [None]*n,
                            sparse_ids if sparse_ids is not None else [None]*n):
        if USE_FAISS and USE_BM25:
            fused = weighted_rrf_fusion(d_ids, s_ids, alpha=HYBRID_ALPHA)
            ranked = [doc_id for doc_id, _ in fused[:top_k_candidates]]
        elif USE_FAISS: ranked = d_ids[:top_k_candidates].tolist()
        elif USE_BM25: ranked = s_ids[:top_k_candidates]
        else: ranked = []
        results.append(ranked)
    return results

def batch_rerank(queries, all_candidates_lists, batch_size=128):
    reranker = get_reranker()
    if reranker is None: return [(cand, np.arange(len(cand), 0, -1)) for cand in all_candidates_lists]
    pairs = []
    for q, cands in zip(queries, all_candidates_lists):
        for c in cands:
            doc_text = train_df.iloc[c]["short_text"]
            pairs.append((q, doc_text))
    if not pairs: return [(cand, np.arange(len(cand), 0, -1)) for cand in all_candidates_lists]
    print("  -> Running CrossEncoder reranker...")
    ce_scores = reranker.predict(pairs, batch_size=batch_size, show_progress_bar=True) 
    results = []
    offset = 0
    for cands in all_candidates_lists:
        n_cand = len(cands)
        scores = ce_scores[offset:offset+n_cand]
        results.append((cands, scores))
        offset += n_cand
    return results

def evaluate_on_test_set_fast(test_df, train_df, k=10, candidate_k=200, gt_col="sub_category", sample_size=300):
    if len(test_df) > sample_size:
        print(f"Lấy mẫu {sample_size} câu từ {len(test_df)} câu trong Test Set...")
        test_sample = test_df.sample(n=sample_size, random_state=SEED).reset_index(drop=True)
    else:
        test_sample = test_df
        
    query_texts = [build_query_text(row) for _, row in test_sample.iterrows()]
    
    # In ra tên kịch bản thay vì chỉ số
    print(f" Đánh giá test set ({len(query_texts)} queries) với candidate là train set ({len(train_df)} items)")
    print(f" Đánh giá Ablation Mode {ABLATION_MODE}: {MODE_NAMES.get(ABLATION_MODE, 'Unknown')}")
    start_time = time.time()

    query_embs = None
    if USE_FAISS or USE_MMR:
        print("Encoding all queries cho FAISS & MMR...")
        query_embs = batch_encode_queries(query_texts)

    print("Retrieval Phase...")
    all_candidates = batch_hybrid_retrieve(query_texts, query_embs, top_k_candidates=candidate_k, ef_search=256)

    print(f"Reranking Phase (Use Reranker: {USE_RERANKER})...")
    rerank_data = batch_rerank(query_texts, all_candidates)

    print("Selecting top-k & Áp dụng MMR...")
    final_candidates = []
    for i, (cands, scores) in enumerate(rerank_data):
        if len(cands) > 0:
            sorted_idx = np.argsort(scores)[::-1]
            sorted_cands = [cands[j] for j in sorted_idx]
            
            if USE_MMR:
                final_ids = apply_mmr(query_embs[i:i+1], sorted_cands[:50], embeddings_train, lambda_mult=MMR_LAMBDA, top_k=k)
                final_candidates.append(final_ids)
            else:
                final_candidates.append(sorted_cands[:k])
        else:
            final_candidates.append([])

    print("Computing metrics...")
    hit_scores, precision_scores, recall_scores, ndcg_scores = [], [], [], []
    details = []
    for query_row, rec_ids in tqdm(zip(test_sample.iterrows(), final_candidates), total=len(test_sample), desc="Evaluating queries", leave=False):
        query_row = query_row[1]
        gt_ids = get_ground_truth(query_row, train_df)

        if not gt_ids: continue
        hit = hit_rate_at_k(rec_ids, gt_ids)
        prec = precision_at_k(rec_ids, gt_ids, k)
        rec = recall_at_k(rec_ids, gt_ids, k)
        ndcg = ndcg_at_k(rec_ids, gt_ids, k)

        hit_scores.append(hit)
        precision_scores.append(prec)
        recall_scores.append(rec)
        ndcg_scores.append(ndcg)

        details.append({
            "query_index": int(query_row.name), "query_title": str(query_row.get("title", "")),
            "query_sub_category": str(query_row[gt_col]), "recommended_ids": [int(x) for x in rec_ids],
            "ground_truth_size": int(len(gt_ids)), f"hit@{k}": float(hit), f"precision@{k}": float(prec),
            f"recall@{k}": float(rec), f"ndcg@{k}": float(ndcg)
        })

    elapsed = time.time() - start_time
    print(f"Evaluation completed in {elapsed:.1f}s")

    results = {
        "ablation_mode": ABLATION_MODE, 
        "ablation_name": MODE_NAMES.get(ABLATION_MODE, 'Unknown'),
        "evaluation_type": "TEST_SET_EVALUATION",
        "k": int(k), "sample_size": int(len(details)), "candidate_k": int(candidate_k),
        f"HitRate@{k}": float(np.mean(hit_scores)) if hit_scores else 0.0,
        f"Precision@{k}": float(np.mean(precision_scores)) if precision_scores else 0.0,
        f"Recall@{k}": float(np.mean(recall_scores)) if recall_scores else 0.0,
        f"NDCG@{k}": float(np.mean(ndcg_scores)) if ndcg_scores else 0.0,
        "details": details
    }
    return results

# ---------- CHẠY ĐÁNH GIÁ ----------
print(f"Đang đánh giá với EVAL_CANDIDATE_K={EVAL_CANDIDATE_K}, EVAL_SAMPLE_SIZE={EVAL_SAMPLE_SIZE}...")
print(f"Đang đánh giá với ABLATION MODE {ABLATION_MODE}: {MODE_NAMES.get(ABLATION_MODE, 'Unknown')} ...")
full_eval_results = evaluate_on_test_set_fast(test_df, train_df, k=EVAL_K, candidate_k=EVAL_CANDIDATE_K, gt_col=EVAL_GT_COL, sample_size=EVAL_SAMPLE_SIZE)

# Lưu JSON và Plot
with open(EVAL_OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(full_eval_results, f, ensure_ascii=False, indent=2)
print(f"💾 Đã lưu {EVAL_OUTPUT_JSON}")

print("     KẾT QUẢ ĐÁNH GIÁ HỆ THỐNG (METRICS)     ")
print(f" 🎯 HitRate@{EVAL_K}:   {full_eval_results[f'HitRate@{EVAL_K}']:.4f}")
print(f" 🎯 Precision@{EVAL_K}: {full_eval_results[f'Precision@{EVAL_K}']:.4f}")
print(f" 🎯 Recall@{EVAL_K}:    {full_eval_results[f'Recall@{EVAL_K}']:.4f}")
print(f" 🎯 NDCG@{EVAL_K}:      {full_eval_results[f'NDCG@{EVAL_K}']:.4f}")

metric_names = [f"HitRate@{EVAL_K}", f"Precision@{EVAL_K}", f"Recall@{EVAL_K}", f"NDCG@{EVAL_K}"]
metric_values = [full_eval_results[m] for m in metric_names]
plt.figure(figsize=(8, 5))
plt.bar(metric_names, metric_values, color=['#4E79A7', '#59A14F', '#F28E2B', '#B07AA1'])
plt.ylim(0, 1)

# Đặt tên biểu đồ theo từ điển
plt.title(f"Evaluation - {MODE_NAMES.get(ABLATION_MODE, 'Unknown')}")

plt.ylabel("Score")
plt.grid(axis="y", alpha=0.3)
for i, v in enumerate(metric_values):
    plt.text(i, v + 0.02, f"{v:.4f}", ha="center", fontweight='bold')
plt.tight_layout()
plt.savefig(EVAL_OUTPUT_FIG, dpi=300)
plt.show()
print(f"💾 Đã lưu biểu đồ {EVAL_OUTPUT_FIG}")

In [ ]:
# ============================================================
# CELL 9: LƯU OUTPUT PARQUET + METADATA
# ============================================================
print("SAVE OUTPUT PARQUET + METADATA".center(110, "="))
for df_name, df in [('train', train_df), ('test', test_df)]:
    if 'price' in df.columns:
        df['price'] = df['price'].astype(str).replace('nan', '').replace('None', '')
        if df_name == 'train':
            train_df = df
        else:
            test_df = df

train_df.to_parquet(f"{SAVE_DIR}/train_final.parquet", index=False)
test_df.to_parquet(f"{SAVE_DIR}/test_final.parquet", index=False)

config_meta = {
    "version": VERSION,
    "data_size": DATA_SIZE,
    "model_name": MODEL_NAME,
    "max_len": MAX_LEN,
    "batch_size_total": BATCH_SIZE_TOTAL,
    "use_8bit": USE_8BIT,
    "top_k_retrieve": TOP_K_RETRIEVE,
    "min_rating": MIN_RATING,
    "hybrid_alpha": HYBRID_ALPHA,
    "train_ratio": TRAIN_RATIO,
    "test_ratio": TEST_RATIO,
    "seed": SEED,
    "candidate_k_eval": EVAL_CANDIDATE_K,
    "eval_sample_size": EVAL_SAMPLE_SIZE,
    "reranker": RERANKER_MODEL,
    "use_mmr": USE_MMR,
    "created_at": datetime.now().isoformat()
}
with open(f"{SAVE_DIR}/config.json", "w") as f:
    json.dump(config_meta, f, indent=2)

save_meta = {
    "saved_at": datetime.now().isoformat(),
    "num_train_products": len(train_df),
    "num_test_products": len(test_df),
    "embedding_shape": list(embeddings_train.shape),
}
with open(f"{SAVE_DIR}/save_meta.json", "w") as f:
    json.dump(save_meta, f, indent=2)

print(f"OUTPUT FILES IN: {SAVE_DIR}")
for f in sorted(os.listdir(SAVE_DIR)):
    size = os.path.getsize(f"{SAVE_DIR}/{f}") / 1024
    unit = "KB" if size < 1024 else "MB"
    size = size if unit == "KB" else size/1024
    print(f"  - {f:<30s} {size:.1f} {unit}")
print("COMPLETE".center(110, "="))

In [ ]:
# ============================================================
# CELL: LOAD FULL PIPELINE FROM KAGGLE INPUT CACHE (OPTIMIZED)
# ============================================================

import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

!pip install -q transformers accelerate faiss-cpu rank-bm25 sentence-transformers
!pip install -q bitsandbytes pyarrow scikit-learn matplotlib pandas numpy tqdm

import os, json, pickle, gc, time
import numpy as np
import pandas as pd
import faiss
import torch
from transformers import AutoTokenizer, AutoModel, BitsAndBytesConfig
from sentence_transformers import CrossEncoder
from rank_bm25 import BM25Okapi

# ---------------- PATH CACHE ----------------
CACHE_DIR = "/kaggle/input/datasets/loilehuu/automotive-cache-mode5"

# ---------------- LOAD CONFIG ----------------
with open(f"{CACHE_DIR}/config.json", "r") as f:
    config = json.load(f)

ABLATION_MODE = config.get("ablation_mode", 5)

USE_FAISS    = ABLATION_MODE >= 2
USE_BM25     = ABLATION_MODE == 1 or ABLATION_MODE >= 3
USE_MMR      = ABLATION_MODE >= 4
USE_RERANKER = ABLATION_MODE == 5

MODEL_NAME = config["model_name"]
RERANKER_MODEL = config["reranker"]
MAX_LEN = config["max_len"]

print("✅ Loaded config:", config)
print("ABLATION_MODE =", ABLATION_MODE)
print(f"FAISS={USE_FAISS} | BM25={USE_BM25} | MMR={USE_MMR} | RERANKER={USE_RERANKER}")

# ---------------- LOAD DATA ----------------
train_df = pd.read_parquet(f"{CACHE_DIR}/train_final.parquet").reset_index(drop=True)
test_df  = pd.read_parquet(f"{CACHE_DIR}/test_final.parquet").reset_index(drop=True)

print(f"📦 Train: {len(train_df)} | Test: {len(test_df)}")

# ---------------- LOAD FAISS ----------------
index = faiss.read_index(f"{CACHE_DIR}/faiss_train.index")
print(f"✅ FAISS loaded: {index.ntotal} vectors")

# ---------------- LOAD BM25 ----------------
with open(f"{CACHE_DIR}/bm25_index.pkl", "rb") as f:
    bm25_index = pickle.load(f)

print("✅ BM25 loaded")

# ---------------- LOAD EMBEDDINGS ----------------
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = hf_token
except:
    print("⚠️ Không lấy được HF_TOKEN, dùng token mặc định nếu có")
    
embeddings_train = np.load(f"{CACHE_DIR}/embeddings_train.npy")
print("✅ Embeddings:", embeddings_train.shape)

# ---------------- TOKENIZER ----------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# ---------------- MODEL (LOAD ONCE) ----------------
def load_query_model():
    bnb_config = BitsAndBytesConfig(
        load_in_8bit=True,
        bnb_8bit_compute_dtype=torch.float16
    )
    model = AutoModel.from_pretrained(
        MODEL_NAME,
        device_map="auto",
        quantization_config=bnb_config,
        dtype=torch.float16
    ).eval()
    return model

print("🔄 Loading query model once...")
query_model = load_query_model()
query_model.eval()

device = next(query_model.parameters()).device

# ---------------- UTIL ----------------
def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).float()
    return (last_hidden_state * mask).sum(1) / mask.sum(1).clamp(min=1e-9)

_query_cache = {}

def encode_query_cached(query):
    if query in _query_cache:
        return _query_cache[query]

    formatted = f"Instruct: Find automotive products.\nQuery: {query}"

    inputs = tokenizer(
        formatted,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=MAX_LEN
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.inference_mode():
        out = query_model(**inputs)

    emb = mean_pool(out.last_hidden_state, inputs["attention_mask"])
    emb = emb.cpu().numpy().astype("float32")
    emb = emb / (np.linalg.norm(emb, axis=1, keepdims=True) + 1e-8)

    _query_cache[query] = emb
    return emb

# ---------------- SEARCH ----------------
def search(query, top_k=5):
    query_emb = encode_query_cached(query)

    D, I = index.search(query_emb, 50)
    candidates = I[0].tolist()

    if USE_RERANKER:
        model = CrossEncoder(RERANKER_MODEL, device="cuda:0" if torch.cuda.is_available() else "cpu")
        pairs = [(query, train_df.iloc[i]["short_text"]) for i in candidates]
        scores = model.predict(pairs)
        candidates = [x for x, _ in sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)]

    return train_df.iloc[candidates[:top_k]][
        ["title", "category", "sub_category", "rating"]
    ]

# ---------------- MULTI QUERY TEST ----------------
import pandas as pd

# Cấu hình Pandas để hiển thị full nội dung
pd.set_option('display.max_colwidth', None)  # Không giới hạn độ dài ký tự trong 1 cột (hiển thị toàn bộ title)
pd.set_option('display.width', 1000)         # Mở rộng chiều ngang của bảng
pd.set_option('display.max_columns', None)   # Hiển thị tất cả các cột nếu có nhiều cột

queries = [
    "high performance brake pads for sports car",
    "engine oil for diesel truck",
    "LED headlight upgrade kit",
    "car interior cleaning spray",
    "portable car jump starter",
    "má phanh hiệu suất cao cho xe thể thao",
    "dầu động cơ cho xe tải diesel",
    "bộ nâng cấp đèn pha LED",
    "bình xịt vệ sinh nội thất ô tô",
    "bộ kích nổ ô tô di động"
]

for q in queries:
    start = time.time()
    print("\n" + "="*80)
    print(f'QUERY: {q}')
    res = search(q, top_k=5)
    display(res)
    print(f"Time: {time.time() - start:.3f}s")

# ============================================================
# INTERACTIVE QUERY
# ============================================================

while True:
    query = input("\nNhập query (gõ 'exit' để thoát): ").strip()

    if query.lower() in ["exit", "quit", "q"]:
        print("Đã thoát.")
        break

    start = time.time()

    print("\n" + "="*80)
    print(f"QUERY: {query}")

    result = search(query, top_k=5)
    display(result)

    print(f"Time: {time.time() - start:.3f}s")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 73.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 92.8 MB/s eta 0:00:00:00:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is inco

config.json:   0%|          | 0.00/729 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...


QUERY: turbo


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-12-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,title,category,sub_category,rating
25793,LFPartS Metal License Plate Frame (Turbo Red),Other,Other,5.0
11712,LFPartS Metal License Plate Frame (Turbo Chrome),Other,Other,5.0
134209,"10X 2.5""Stainless Steel T-BOLT CLAMPS Turbo Piping Hose",Other,Other,5.0
197124,Lyman Turbo Sonic Cleaner Promo Pack,Car Care,Car Care,5.0
179881,RZR,Other,Other,5.0


Time: 3.133s


In [ ]:
#